# Indicator Lab\n\n这个 Notebook 用于完成指标白盒分析。目标不是只画图，而是保留从原始数据到指标计算再到图表展示的完整过程。

In [ ]:
import pandas as pd\nimport numpy as np\nimport matplotlib.pyplot as plt\n\nplt.style.use('seaborn-v0_8-whitegrid')

In [ ]:
jushi = pd.read_csv('../raw_data/china-jushi_600176.csv', parse_dates=['date'])\nzhongji = pd.read_csv('../raw_data/zhongji-xuchuang_300308.csv', parse_dates=['date'])\n\nprint('中国巨石:', jushi.shape)\nprint('中际旭创:', zhongji.shape)\njushi.head()

In [ ]:
def add_ma(df, windows=(5, 10, 20, 60)):\n    out = df.copy()\n    for w in windows:\n        out[f'MA{w}'] = out['close'].rolling(w).mean()\n    return out\n\ndef add_macd(df, fast=12, slow=26, signal=9):\n    out = df.copy()\n    ema_fast = out['close'].ewm(span=fast, adjust=False).mean()\n    ema_slow = out['close'].ewm(span=slow, adjust=False).mean()\n    out['DIF'] = ema_fast - ema_slow\n    out['DEA'] = out['DIF'].ewm(span=signal, adjust=False).mean()\n    out['MACD_Hist'] = out['DIF'] - out['DEA']\n    return out\n\ndef add_bollinger(df, window=20, k=2):\n    out = df.copy()\n    mid = out['close'].rolling(window).mean()\n    std = out['close'].rolling(window).std()\n    out['BB_Middle'] = mid\n    out['BB_Upper'] = mid + k * std\n    out['BB_Lower'] = mid - k * std\n    out['BB_Width_Pct'] = (out['BB_Upper'] - out['BB_Lower']) / out['BB_Middle'] * 100\n    return out\n\ndef add_rsi(df, window=14):\n    out = df.copy()\n    delta = out['close'].diff()\n    gain = delta.clip(lower=0)\n    loss = -delta.clip(upper=0)\n    avg_gain = gain.rolling(window).mean()\n    avg_loss = loss.rolling(window).mean()\n    rs = avg_gain / avg_loss\n    out['RSI'] = 100 - (100 / (1 + rs))\n    return out\n\ndef add_amplitude(df):\n    out = df.copy()\n    prev_close = out['close'].shift(1)\n    out['Amplitude_Pct'] = (out['high'] - out['low']) / prev_close * 100\n    return out\n

In [ ]:
df = jushi.copy()\ndf = add_ma(df)\ndf = add_macd(df)\ndf = add_bollinger(df)\ndf = add_rsi(df)\ndf = add_amplitude(df)\n\ndf.tail()

In [ ]:
fig, ax = plt.subplots(figsize=(14, 6))\nax.plot(df['date'], df['close'], label='Close', color='black', linewidth=1.3)\nax.plot(df['date'], df['MA5'], label='MA5', linewidth=1)\nax.plot(df['date'], df['MA10'], label='MA10', linewidth=1)\nax.plot(df['date'], df['MA20'], label='MA20', linewidth=1.2)\nax.plot(df['date'], df['BB_Upper'], label='BOLL Upper', linestyle='--', alpha=0.8)\nax.plot(df['date'], df['BB_Middle'], label='BOLL Middle', linestyle='--', alpha=0.8)\nax.plot(df['date'], df['BB_Lower'], label='BOLL Lower', linestyle='--', alpha=0.8)\nax.set_title('Close Price, Moving Averages and Bollinger Bands')\nax.legend(ncol=4)\nplt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(14, 4))\nax.plot(df['date'], df['RSI'], label='RSI(14)', color='royalblue')\nax.axhline(70, linestyle='--', color='red', alpha=0.6)\nax.axhline(30, linestyle='--', color='green', alpha=0.6)\nax.set_title('RSI')\nax.legend()\nplt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(14, 4))\ncolors = ['#ef4444' if x >= 0 else '#93c5fd' for x in df['MACD_Hist'].fillna(0)]\nax.bar(df['date'], df['MACD_Hist'], color=colors, alpha=0.7, label='MACD Hist')\nax.plot(df['date'], df['DIF'], label='DIF', color='firebrick')\nax.plot(df['date'], df['DEA'], label='DEA', color='steelblue')\nax.set_title('MACD')\nax.legend()\nplt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(14, 4))\nax.plot(df['date'], df['Amplitude_Pct'], color='purple', label='Amplitude %')\nax.set_title('Daily Amplitude')\nax.legend()\nplt.show()

## 观察记录\n\n这里建议你补充：\n\n1. 当前价格位于哪些均线上方/下方\n2. 布林带是否被突破\n3. RSI 是否接近超买/超卖\n4. MACD 当前红柱还是绿柱\n5. 最近振幅是否明显放大